# Scheduling & Monitoring Job in Production

## 1. Scheduling the Job

### Configure the schedule
- In the **Jobs UI**, open the job and go to **Schedule** (or `Triggers` for multi-task jobs).
- Choose a **Scheduled** trigger with a cron expression (e.g. `0 0 2 * * *` for daily at 02:00), a timezone, and whether to **pause** the schedule.
- Other trigger options for production:
  - **File arrival** — run when new files land in a UC volume or external location.
  - **Continuous / development** — for streaming-style always-running tasks.
  - **Run job trigger** — chain this job after an upstream job finishes.


### Scheduling best practices
- **Concurrency control** — set *Maximum concurrent runs* (usually `1` for pipelines that write to the same tables) so overlapping runs don't corrupt data.
- **Retries** — configure *On failure* retry policy (e.g. 2 retries with a few minutes between attempts) to ride out transient cluster/cloud errors.
- **Timeout** — always set a run timeout so a hung run can't burn compute indefinitely.
- **Alerts on schedule** — add email/webhook/Slack notifications for *Run start failure*, *Run failure*, and *Run success* (success only while stabilizing a new deployment).
- **Stagger schedules** — if this job feeds downstream jobs, or shares a warehouse/cluster pool, offset its cron from neighbors to avoid resource contention.
- **Avoid DST surprises** — pick a timezone and confirm cron behavior across daylight-saving transitions.

---

## 2. Monitoring the Job

### a) Jobs UI
- **Runs tab** — per-run status, duration, tasks, and the run's **Job Run** page (task graphs, logs, error messages).
- Set notification subscriptions at the job level so the team is paged automatically rather than relying on someone watching the UI.

### b) System tables (account-wide, SQL-based)

**Run history & result states** — `system.lakeflow.job_run_timeline` (or `job_run_timeline_v2` where enabled):

```sql
SELECT
  period,
  run_id,
  result_state,
  trigger_type,
  start_time,
  end_time,
  task_run_count
FROM system.lakeflow.job_run_timeline
WHERE job_id = 463795192221857
  AND period >= DATE_SUB(CURRENT_DATE(), 7)
ORDER BY start_time DESC;
```

**Recent failures & duration trend** (SQL alert candidates):

```sql
SELECT
  TO_CHAR(DATE(start_time), 'yyyy-MM-dd') AS run_date,
  COUNT(*) AS runs,
  SUM(CASE WHEN result_state = 'FAILED' THEN 1 ELSE 0 END) AS failures,
  ROUND(AVG(duration_seconds), 1) AS avg_duration_sec,
  PERCENTILE(duration_seconds, 0.95) AS p95_duration_sec
FROM (
  SELECT *, TIMESTAMP_DIFF(SECOND, start_time, end_time) AS duration_seconds
  FROM system.lakeflow.job_run_timeline
  WHERE job_id = 463795192221857
    AND period >= DATE_SUB(CURRENT_DATE(), 30)
)
GROUP BY run_date
ORDER BY run_date DESC;
```

**Job definition (SCD2 — always dedupe to the latest version):**

```sql
SELECT *
FROM system.lakeflow.jobs
WHERE job_id = 463795192221857
QUALIFY ROW_NUMBER() OVER (PARTITION BY workspace_id, job_id
                           ORDER BY change_time DESC) = 1;
```
> Note: `jobs` is an SCD2 table — without the `QUALIFY` filter you get one row per edit, not the current definition.

**Cost / DBU attribution** — `system.billing.usage` tagged by `job_id`:

```sql
SELECT
  usage_date,
  sku_name,
  SUM(usage_quantity) AS dbus,
  SUM(usage_quantity * list_pricing.pricing.default_list_price) AS est_cost_usd
FROM system.billing.usage
LEFT JOIN system.billing.list_prices list_pricing
  ON usage.sku_name = list_pricing.sku_name
 AND usage.usage_end_time >= list_pricing.price_start_time
 AND usage.usage_end_time <  list_pricing.price_end_time
WHERE usage_metadata.job_id = '463795192221857'
  AND usage_date >= DATE_SUB(CURRENT_DATE(), 30)
GROUP BY usage_date, sku_name
ORDER BY usage_date DESC;
```

### c) SQL Alerts (automated monitoring)
Wrap the failure query above in a **SQL Alert** that fires when, for example:
- any run in the last 24h ended in `FAILED` / `TIMEDOUT` / `CANCELED`,
- the p95 run duration increased by more than 50% week-over-week (regression / data growth),
- zero successful runs in the expected daily window (missed schedule — the schedule was paused or the trigger broke).

Route alert notifications to email, Slack/Teams, or a webhook that opens a ticket.

---

## 3. Production Checklist
| Concern | Action |
|---|---|
| Schedule | Cron + timezone defined, paused only intentionally |
| Overlap | Max concurrent runs = 1 (or validated safe concurrency) |
| Transient failures | Retry policy + alert on repeated failures |
| Hung runs | Run timeout configured |
| Alerting | Job notifications **and** SQL alerts on system tables |
| Drift | Weekly review of duration trend and DBU cost |
| Ownership | `owner` field set to a team principal, not an individual |
